In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'/ 'raw'

train_df = pd.read_csv(DATA_DIR / 'train.csv')

label_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]


print("Dataset loaded:", train_df.shape )
print("Project root:", PROJECT_ROOT)

Dataset loaded: (159571, 8)
Project root: d:\Academics\TrustLens


In [3]:
x= train_df['comment_text']
y= train_df[label_columns]

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_indices, validation_indices = next(splitter.split(x,y))

train_data= train_df.iloc[train_indices].copy()
validation_data= train_df.iloc[validation_indices].copy()

print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))

Training rows: 127656
Validation rows: 31915


In [6]:
distribution = pd.DataFrame({

    'Full Dataset': train_df[label_columns].mean() *100,
    'Traning': train_data[label_columns].mean() *100,
    'Validation': validation_data[label_columns].mean() *100
    
})

display(distribution.round(3))

,Full Dataset,Traning,Validation
toxic,9.584,9.584,9.585
severe_toxic,1.000,1.000,1.000
obscene,5.295,5.295,5.295
threat,0.300,0.299,0.301
insult,4.936,4.937,4.935
identity_hate,0.880,0.880,0.880


In [7]:
overlap = set(train_data.index).intersection(
    set(validation_data.index)
)

print("Index overlap:", len(overlap))

Index overlap: 0


In [8]:
import html
import re

def normalize_comment(text):
    """
    Conservative normalization:
    - Decode HTML entities
    - Normalize Unicode-compatible text through HTML decoding
    - Replace repeated whitespace with a single space
    - Preserve punctuation, emojis, URLs, and capitalization
    """
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
examples = [
    "Hello&nbsp;&nbsp;world!",
    "YOU ARE AN IDIOT!!!",
    "Visit https://example.com",
    "I can't believe this 😡"
]

for example in examples:
    print("Original:  ", example)
    print("Normalized:", normalize_comment(example))
    print("-" * 60)

Original:   Hello&nbsp;&nbsp;world!
Normalized: Hello world!
------------------------------------------------------------
Original:   YOU ARE AN IDIOT!!!
Normalized: YOU ARE AN IDIOT!!!
------------------------------------------------------------
Original:   Visit https://example.com
Normalized: Visit https://example.com
------------------------------------------------------------
Original:   I can't believe this 😡
Normalized: I can't believe this 😡
------------------------------------------------------------


In [10]:
train_data["normalized_text"] = train_data["comment_text"].apply(
    normalize_comment
)

validation_data["normalized_text"] = validation_data["comment_text"].apply(
    normalize_comment
)

print("Training text normalization complete.")
print("Validation text normalization complete.")

Training text normalization complete.
Validation text normalization complete.


In [11]:
print(
    "Empty training comments:",
    (train_data["normalized_text"].str.len() == 0).sum()
)

print(
    "Empty validation comments:",
    (validation_data["normalized_text"].str.len() == 0).sum()
)

Empty training comments: 0
Empty validation comments: 0


In [12]:
comparison = train_data[
    ["comment_text", "normalized_text"]
].sample(10, random_state=42)

display(comparison)

,comment_text,normalized_text
111408,people on wikipedia think they know everything...,people on wikipedia think they know everything...
37138,"Well, sure. I'd be glad to help someone, And I...","Well, sure. I'd be glad to help someone, And I..."
4536,"As I've explained in the article, the noise is...","As I've explained in the article, the noise is..."
35106,"Haha I'm not really one to genrewank, but Wint...","Haha I'm not really one to genrewank, but Wint..."
148138,Rebecca Cummings article \nYou have previously...,Rebecca Cummings article You have previously h...
4890,"""\n\nRense\nWhy is Tom Harrison posting articl...",""" Rense Why is Tom Harrison posting articles f..."
128643,"""\n\nYou seem to think that references to cano...",""" You seem to think that references to canon S..."
124129,Laika \n\nJust when I thought the article coul...,Laika Just when I thought the article could no...
66121,"Sorry to see you go \n\nHal, I'm sorry to hear...","Sorry to see you go Hal, I'm sorry to hear you..."
64985,Otpor!\nMy bad. My browser crashed during edit...,Otpor! My bad. My browser crashed during editi...


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=300_000,
    sublinear_tf=True,
    dtype=np.float32
)

X_train_word = word_vectorizer.fit_transform(
    train_data["normalized_text"]
)

X_validation_word = word_vectorizer.transform(
    validation_data["normalized_text"]
)

print("Training feature shape:", X_train_word.shape)
print("Validation feature shape:", X_validation_word.shape)
print("Vocabulary size:", len(word_vectorizer.vocabulary_))

Training feature shape: (127656, 300000)
Validation feature shape: (31915, 300000)
Vocabulary size: 300000


In [17]:
print("Training matrix:")
print(X_train_word)

print("\nValidation matrix:")
print(X_validation_word)

print("\nTraining matrix type:", type(X_train_word))
print("Training matrix data type:", X_train_word.dtype)

Training matrix:
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 10975736 stored elements and shape (127656, 300000)>
  Coords	Values
  (0, 90588)	0.09560639411211014
  (0, 288687)	0.057459913194179535
  (0, 246940)	0.04901014640927315
  (0, 81820)	0.06791400164365768
  (0, 152001)	0.06622078269720078
  (0, 269639)	0.07350371032953262
  (0, 164347)	0.04570392146706581
  (0, 274077)	0.09757845103740692
  (0, 110141)	0.1392170786857605
  (0, 159591)	0.15993347764015198
  (0, 92912)	0.10624630749225616
  (0, 284612)	0.061427753418684006
  (0, 208882)	0.08221027255058289
  (0, 257309)	0.05300676450133324
  (0, 285354)	0.11547844111919403
  (0, 275725)	0.1567738652229309
  (0, 138795)	0.04968607425689697
  (0, 59968)	0.14408746361732483
  (0, 179571)	0.036208175122737885
  (0, 226867)	0.05375773087143898
  (0, 104002)	0.12734512984752655
  (0, 13014)	0.06879408657550812
  (0, 278977)	0.12301793694496155
  (0, 35392)	0.045959293842315674
  (0, 167616)	0.06676807999610901
  :	:


In [18]:
print("Vectorizer fitted on training data only:", X_train_word.shape[1] == X_validation_word.shape[1])

Vectorizer fitted on training data only: True


In [19]:
print("train_data exists:", "train_data" in globals())
print("validation_data exists:", "validation_data" in globals())
print("X_train_word exists:", "X_train_word" in globals())
print("X_validation_word exists:", "X_validation_word" in globals())

train_data exists: True
validation_data exists: True
X_train_word exists: True
X_validation_word exists: True


In [20]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

# Separate the six target labels
y_train = train_data[label_columns].values
y_validation = validation_data[label_columns].values

print("Training labels shape:", y_train.shape)
print("Validation labels shape:", y_validation.shape)

Training labels shape: (127656, 6)
Validation labels shape: (31915, 6)


In [24]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

baseline_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ),
    n_jobs=1  # Avoid joblib multiprocessing
)

baseline_model.fit(X_train_word, y_train)

print("Baseline model training complete.")

Baseline model training complete.


In [25]:
print("Train matrix:")
print("Shape:", X_train_word.shape)
print("Type:", type(X_train_word))
print("Data writable:", X_train_word.data.flags.writeable)
print("Indices writable:", X_train_word.indices.flags.writeable)
print("Indptr writable:", X_train_word.indptr.flags.writeable)

Train matrix:
Shape: (127656, 300000)
Type: <class 'scipy.sparse._csr.csr_matrix'>
Data writable: True
Indices writable: True
Indptr writable: True


In [26]:
import scipy
import sklearn
import numpy as np

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

NumPy: 2.5.4
SciPy: 1.18.1
scikit-learn: 1.9.1


In [27]:
X_train_model = X_train_word.copy()
X_validation_model = X_validation_word.copy()

X_train_model.data = X_train_model.data.copy()
X_train_model.indices = X_train_model.indices.copy()
X_train_model.indptr = X_train_model.indptr.copy()

print("Train copy writable:", X_train_model.data.flags.writeable)
print("Validation copy writable:", X_validation_model.data.flags.writeable)

Train copy writable: True
Validation copy writable: True


In [28]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

baseline_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ),
    n_jobs=1
)

baseline_model.fit(X_train_model, y_train)

print("Baseline model training complete.")

Baseline model training complete.


In [29]:
validation_probabilities = baseline_model.predict_proba(X_validation_model)

print("Probability matrix shape:", validation_probabilities.shape)
print("First comment probabilities:", validation_probabilities[0])

Probability matrix shape: (31915, 6)
First comment probabilities: [0.03862609 0.0019943  0.0106455  0.00106195 0.01546778 0.00286544]


In [30]:
from sklearn.metrics import classification_report, f1_score

validation_predictions = (validation_probabilities >= 0.5).astype(int)

print("Micro F1:", f1_score(y_validation, validation_predictions, average="micro", zero_division=0))
print("Macro F1:", f1_score(y_validation, validation_predictions, average="macro", zero_division=0))
print("Weighted F1:", f1_score(y_validation, validation_predictions, average="weighted", zero_division=0))

print("\nPer-label report:")
print(
    classification_report(
        y_validation,
        validation_predictions,
        target_names=label_columns,
        zero_division=0
    )
)

Micro F1: 0.6494074749316318
Macro F1: 0.45227911257708125
Weighted F1: 0.6407939871052138

Per-label report:
               precision    recall  f1-score   support

        toxic       0.94      0.55      0.69      3059
 severe_toxic       0.58      0.25      0.35       319
      obscene       0.94      0.58      0.72      1690
       threat       0.88      0.07      0.13        96
       insult       0.85      0.50      0.63      1575
identity_hate       0.63      0.11      0.19       281

    micro avg       0.90      0.51      0.65      7020
    macro avg       0.80      0.34      0.45      7020
 weighted avg       0.89      0.51      0.64      7020
  samples avg       0.05      0.04      0.04      7020



In [31]:
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import numpy as np

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]

threshold_results = []

for label_index, label_name in enumerate(label_columns):
    for threshold in thresholds:
        predictions = (
            validation_probabilities[:, label_index] >= threshold
        ).astype(int)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_validation[:, label_index],
            predictions,
            average="binary",
            zero_division=0
        )

        threshold_results.append({
            "label": label_name,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(3)

,label,threshold,precision,recall,f1
0,toxic,0.1,0.515,0.918,0.660
1,toxic,0.2,0.730,0.797,0.762
2,toxic,0.3,0.835,0.697,0.760
3,toxic,0.4,0.900,0.613,0.729
4,toxic,0.5,0.938,0.549,0.692
5,severe_toxic,0.1,0.388,0.630,0.480
6,severe_toxic,0.2,0.476,0.495,0.485
7,severe_toxic,0.3,0.516,0.395,0.448
8,severe_toxic,0.4,0.549,0.317,0.402
9,severe_toxic,0.5,0.579,0.254,0.353


In [32]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

threshold_tuning_results = []

for label_index, label_name in enumerate(label_columns):
    precision, recall, thresholds = precision_recall_curve(
        y_validation[:, label_index],
        validation_probabilities[:, label_index]
    )

    # The final precision/recall point has no corresponding threshold.
    precision = precision[:-1]
    recall = recall[:-1]

    f1_scores = (
        2 * precision * recall /
        (precision + recall + 1e-12)
    )

    best_index = np.argmax(f1_scores)

    threshold_tuning_results.append({
        "label": label_name,
        "best_threshold": thresholds[best_index],
        "precision": precision[best_index],
        "recall": recall[best_index],
        "f1": f1_scores[best_index]
    })

best_threshold_df = pd.DataFrame(threshold_tuning_results)

print(best_threshold_df.round(3).to_string(index=False))

        label  best_threshold  precision  recall    f1
        toxic           0.253      0.802   0.745 0.772
 severe_toxic           0.207      0.494   0.492 0.493
      obscene           0.178      0.821   0.791 0.805
       threat           0.085      0.537   0.375 0.442
       insult           0.186      0.689   0.736 0.711
identity_hate           0.128      0.446   0.409 0.427


In [33]:
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Thresholds selected using the validation set
best_thresholds = best_threshold_df["best_threshold"].to_numpy()

# Apply each label's own threshold
tuned_validation_predictions = (
    validation_probabilities >= best_thresholds
).astype(int)

print(
    "Tuned Micro F1:",
    round(
        f1_score(
            y_validation,
            tuned_validation_predictions,
            average="micro",
            zero_division=0
        ),
        3
    )
)

print(
    "Tuned Macro F1:",
    round(
        f1_score(
            y_validation,
            tuned_validation_predictions,
            average="macro",
            zero_division=0
        ),
        3
    )
)

print("\nTuned per-label report:")
print(
    classification_report(
        y_validation,
        tuned_validation_predictions,
        target_names=label_columns,
        zero_division=0
    )
)

Tuned Micro F1: 0.736
Tuned Macro F1: 0.608

Tuned per-label report:
               precision    recall  f1-score   support

        toxic       0.80      0.74      0.77      3059
 severe_toxic       0.49      0.49      0.49       319
      obscene       0.82      0.79      0.81      1690
       threat       0.54      0.38      0.44        96
       insult       0.69      0.74      0.71      1575
identity_hate       0.45      0.41      0.43       281

    micro avg       0.75      0.72      0.74      7020
    macro avg       0.63      0.59      0.61      7020
 weighted avg       0.75      0.72      0.74      7020
  samples avg       0.06      0.07      0.06      7020



In [34]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

weighted_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        class_weight="balanced",
        random_state=42
    ),
    n_jobs=1
)

weighted_model.fit(X_train_model, y_train)

print("Class-weighted model training complete.")

Class-weighted model training complete.


In [35]:
weighted_validation_probabilities = weighted_model.predict_proba(
    X_validation_model
)

print(
    "Weighted probability matrix shape:",
    weighted_validation_probabilities.shape
)

Weighted probability matrix shape: (31915, 6)


In [36]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

weighted_threshold_results = []

for label_index, label_name in enumerate(label_columns):
    precision, recall, thresholds = precision_recall_curve(
        y_validation[:, label_index],
        weighted_validation_probabilities[:, label_index]
    )

    precision = precision[:-1]
    recall = recall[:-1]

    f1_scores = (
        2 * precision * recall /
        (precision + recall + 1e-12)
    )

    best_index = np.argmax(f1_scores)

    weighted_threshold_results.append({
        "label": label_name,
        "best_threshold": thresholds[best_index],
        "precision": precision[best_index],
        "recall": recall[best_index],
        "f1": f1_scores[best_index]
    })

weighted_threshold_df = pd.DataFrame(weighted_threshold_results)

print(weighted_threshold_df.round(3).to_string(index=False))

        label  best_threshold  precision  recall    f1
        toxic           0.609      0.763   0.791 0.777
 severe_toxic           0.907      0.447   0.611 0.517
      obscene           0.680      0.830   0.796 0.813
       threat           0.880      0.476   0.406 0.438
       insult           0.743      0.717   0.724 0.721
identity_hate           0.838      0.440   0.480 0.459


In [37]:
from sklearn.metrics import classification_report, f1_score

weighted_best_thresholds = (
    weighted_threshold_df["best_threshold"].to_numpy()
)

weighted_tuned_predictions = (
    weighted_validation_probabilities >= weighted_best_thresholds
).astype(int)

print(
    "Weighted tuned Micro F1:",
    round(
        f1_score(
            y_validation,
            weighted_tuned_predictions,
            average="micro",
            zero_division=0
        ),
        3
    )
)

print(
    "Weighted tuned Macro F1:",
    round(
        f1_score(
            y_validation,
            weighted_tuned_predictions,
            average="macro",
            zero_division=0
        ),
        3
    )
)

print("\nWeighted tuned per-label report:")
print(
    classification_report(
        y_validation,
        weighted_tuned_predictions,
        target_names=label_columns,
        zero_division=0
    )
)

Weighted tuned Micro F1: 0.742
Weighted tuned Macro F1: 0.621

Weighted tuned per-label report:
               precision    recall  f1-score   support

        toxic       0.76      0.79      0.78      3059
 severe_toxic       0.45      0.61      0.52       319
      obscene       0.83      0.80      0.81      1690
       threat       0.48      0.41      0.44        96
       insult       0.72      0.72      0.72      1575
identity_hate       0.44      0.48      0.46       281

    micro avg       0.73      0.75      0.74      7020
    macro avg       0.61      0.64      0.62      7020
 weighted avg       0.74      0.75      0.74      7020
  samples avg       0.07      0.07      0.07      7020



In [38]:
from pathlib import Path
import joblib
import json

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "trustlens_logistic_weighted.joblib"
threshold_path = models_dir / "trustlens_logistic_weighted_thresholds.json"

joblib.dump(weighted_model, model_path)

threshold_data = {
    label: float(threshold)
    for label, threshold in zip(
        label_columns,
        weighted_best_thresholds
    )
}

with open(threshold_path, "w", encoding="utf-8") as file:
    json.dump(threshold_data, file, indent=4)

print("Model saved to:", model_path.resolve())
print("Thresholds saved to:", threshold_path.resolve())

Model saved to: D:\Academics\TrustLens\models\trustlens_logistic_weighted.joblib
Thresholds saved to: D:\Academics\TrustLens\models\trustlens_logistic_weighted_thresholds.json


In [39]:
from pathlib import Path

model_path = Path("../models/trustlens_logistic_weighted.joblib")
threshold_path = Path("../models/trustlens_logistic_weighted_thresholds.json")

print(f"Model size: {model_path.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Threshold file size: {threshold_path.stat().st_size / 1024:.2f} KB")

Model size: 13.74 MB
Threshold file size: 0.22 KB
